In [ ]:
#Imports generales
import math
import numpy as np
import random

#Imports gráficos
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib import rc
import re

#Imports AG
import sys
!{sys.executable} -m pip install deap
import deap
from deap import base, creator, tools

#Imports AC
import cellpylib as cpl

#Imports paralelización
#import multiprocessing
#from multiprocessing import Pool
from joblib import Parallel, delayed
import time
import os

#Metricas de similitd
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import mean_squared_error


Parámetros del AC
NUM_STATES -> Número de estados de cada célula
VEC_SIZE -> Vecinos considerados a cada lado de la célula
CA_NUM -> Número de autómatas utilizados en la función de fitness para testear la regla
CA_SIZE -> Tamaño del autómata
CA_TIMESTEPS -> Pasos de evolución de los autómatas

Parámetros del AG
IND_SIZE -> Tamaño de los individuos
POP_SIZE -> Tamaño de la poblacion inicial
CXPB -> Probabilidad de cruce
MUTPB -> Probabilidad de mutacion
NGEN -> Número de generaciones
STOP_CONDITION -> Valor de fitness exigido en la condición de parada
FITNESS_THRESHOLD -> Valor de fitness umbral de la función de fitness adaptativa
NUM_MUT -> Número de mutaciones realizadas en cada individuo

In [2]:
NUM_STATES = 3
VEC_SIZE = 1
CA_NUM = 30
CA_SIZE_1 = 18
CA_SIZE_2 = 30
CA_TIMESTEPS = 75
P = 1
MOORE = False
NEUMANN = True

if NEUMANN:
    INPUT_SIZE = 5
elif MOORE:
    INPUT_SIZE = 9
else:
    INPUT_SIZE = 2*VEC_SIZE + 1

IND_SIZE = int(pow(NUM_STATES, INPUT_SIZE))
POP_SIZE = 300
CXPB, MUTPB, NGEN = 0.9, 1, 1000


STOP_CONDITION = 0.8
FITNESS_THRESHOLD = 0.5

NUM_MUT = 10 #Número de genes mutados en cada individuo (implementación de mutaación personalizada en función principal)



'''Patrón final deseado (bandera húngara)'''
HUN = np.zeros((CA_SIZE_1, CA_SIZE_2), dtype=int)
h = CA_SIZE_1 // 3
HUN[0:h, :] = 0     
HUN[h:2*h, :] = 1   
HUN[2*h:, :] = 2

'''Patrón final deseado (bandera austriaca)'''
AUS = np.zeros((CA_SIZE_1, CA_SIZE_2), dtype=int)
h = CA_SIZE_1 // 3
AUS[0:h, :] = 0     
AUS[h:2*h, :] = 1   
AUS[2*h:, :] = 0

'''Patrón final deseado (bandera francesa)'''
FRA = np.zeros((CA_SIZE_1, CA_SIZE_2), dtype=int)
h = CA_SIZE_1 // 3
FRA[:, 0:h] = 0     
FRA[:, h:2*h] = 1   
FRA[:, 2*h:] = 2

'''Patrón final deseado (bandera japonesa)'''
JPN = np.ones((CA_SIZE_1, CA_SIZE_2), dtype=int)
cy, cx = CA_SIZE_1 // 2, CA_SIZE_2 // 2
r = CA_SIZE_1 // 3  # El radio es aprox un tercio de la altura
y, x = np.ogrid[:CA_SIZE_1, :CA_SIZE_2]
mascara_circulo = (x - cx)**2 + (y - cy)**2 <= r**2
JPN[mascara_circulo] = 0



            

'''Creador de fitness y de individuo
#Fitness con único objetivo (weights = (1.0,) y máximo'''
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

'''Especificación de genes, individuos y poblacion
Individuos ternarios, attr_int entre 0 y 2, y en forma de lista
'''
toolbox = base.Toolbox()
toolbox.register("attr_int", random.randint, 0, 2)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_int, IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

'''Regla identidad por si se desea hacer el autómata probabilistico'''
def identity_rule(IND_SIZE):
    rule = [0]*IND_SIZE
    for i in range(IND_SIZE):
        if (NEUMANN):
            ternario = entero_a_base3(i, longitud_total=5)
            rule[i] = int(ternario[0])
        elif (MOORE):
            ternario = entero_a_base3(i, longitud_total=9)
            rule[i] = int(ternario[0])
        else:
            ternario = entero_a_base3(i, longitud_total=2*VEC_SIZE + 1)
            rule[i] = int(ternario[VEC_SIZE])
    return rule
        

'''Creador de regla de transición en el formato requerido por CellPyLib a partir de individuo'''
def create_transition_rule(individual, probability, ca_size_1, ca_size_2):
    rule_table = np.array(individual, dtype=int)
    id_rule_table = np.array(identity_rule(len(individual)), dtype=int)
    powers = np.array([81, 27, 9, 3, 1], dtype=int) #Para von neumann
    def my_rule(cells, r, t):
        if r[0] == 0: 
            return 0
        if r[0] == ca_size_1 - 1: 
            return 2
        '''if r[1] == 0 or r[1] == ca_size_2 - 1:
            if r[0] < ca_size_1//3:
                return 0
            elif r[0] >= 2*ca_size_1//3:
                return 2
            else: 
                return 1'''
        
        neighbors = np.array([
            cells[1, 1], # Center
            cells[0, 1], # Up
            cells[1, 2], # Right
            cells[2, 1], # Down
            cells[1, 0]  # Left
        ], dtype=int)
        
        '''Calculo del indice  de la regla a partir de la configuración del vecindario en base 3'''
        if NEUMANN:
            idx = np.dot(neighbors, powers) 
        elif MOORE:
            clean_cells = np.array(cells).flatten().astype(int)
            key = ''.join(str(c) for c in clean_cells)

        '''Devolvemos el nuevo estado de la célula central según el índice de la regla calculado previamente'''
        if random.random() < probability: 
            return rule_table[idx]
        else: 
            return id_rule_table[idx]  
            
    return my_rule

'''Generación aleatoria de autómatas, teniendo en cuenta la frontera fija'''
def generate_CAs(ca_num, num_states, ca_size_1, ca_size_2):
    CAs = []
    for i in range(ca_num):
        ca = np.random.randint(0, NUM_STATES, size=(ca_size_1, ca_size_2))
        '''Fila superior e inferior fijas'''
        ca[0, :] = 0
        ca[-1, :] = 2
        ca = np.expand_dims(ca, axis=0)
        CAs.append(ca)
    return CAs

'''Conversión de enteros a base3, necesaria para determinar los vecindarios asociados a cada índice de las reglas'''
def entero_a_base3(numero, longitud_total=0):
    if numero == 0:
        return "0".zfill(longitud_total)

    ternario = ""
    n = numero

    while n > 0:
        residuo = n % 3
        ternario = str(residuo) + ternario 
        n = n // 3

    return ternario.zfill(longitud_total)

'''Creación del diccionario de reglas. Recibe un individuo (array de elementos ternarios), y para cada elemento
calcula su índice en ternario (vecindario) y le asigna el nuevo estado de la célula central'''
def gen_rule_dict(individual):
    rule_dict = {}
    for i in range(len(individual)):
        if (NEUMANN):
            ternario = entero_a_base3(i, longitud_total=5)
            rule_dict[ternario] = individual[i]
        elif (MOORE):
            ternario = entero_a_base3(i, longitud_total=9)
            rule_dict[ternario] = individual[i]
        else:
            ternario = entero_a_base3(i, longitud_total=2*VEC_SIZE + 1)
            rule_dict[ternario] = individual[i]

    return rule_dict

'''Evolucion de los autómatas de la lista CAs mediante mi_regla'''
def evolved_CAs(CAs, mi_regla):
    evolved_CAs = []
    for i in range(len(CAs)):
        evolved_CA = cpl.evolve2d(CAs[i], timesteps=CA_TIMESTEPS, neighbourhood = "von Neumann", apply_rule=mi_regla)
        evolved_CAs.append(evolved_CA)

    return evolved_CAs

'''Metrica accuracy'''
def accuracy_metric(ca_final, target):
    return (np.sum(ca_final == target)/(CA_SIZE_1*CA_SIZE_2))

'''Metrica Jaccard'''
def jaccard_metric(ca_final, target):
    jaccard_states = []
    for state in range(NUM_STATES):
        intersection = np.sum((ca_final == state) & (target == state))
        union = np.sum((target == state) | (ca_final == state))
        if union == 0:
            jaccard_states.append(1.0)  # Si no hay elementos en la unión, consideramos Jaccard como 1
        else:
            jaccard_states.append(intersection / union)
    return np.mean(jaccard_states)

'''Métrica SSIM'''
def ssim_metric(ca_final, target):
    return ssim(ca_final, target, data_range = NUM_STATES - 1, win_size=5) 

'''Métrica MSE'''
def mse_metric(ca_final, target):
    return mean_squared_error(ca_final, target, data_range = NUM_STATES - 1) 

'''Metrica Correlation'''
def correlation_metric(ca_final, target):
    ca_mean = np.mean(ca_final)
    target_mean = np.mean(target)
    
    covariance = np.sum((ca_final - ca_mean) * (target - target_mean))
    variance_ca = np.sum((ca_final - ca_mean)**2)
    variance_target = np.sum((target - target_mean)**2)
    
    if variance_ca == 0 or variance_target == 0:
        return 0
    else:
        return covariance / np.sqrt(variance_ca * variance_target)

'''Metrica Haussdorff'''
'''ESTUDIAR SI AÑADIR MAS PARAMETROS, UMBRALIZACION, SEGMENTACION, COMO MODELAR...'''
def haussdorf_metric(ca_final, target):
    return hausdorff_distance(ca_final, target) 



'''Funcion que genera la regla a partir de un individuo, evoluciona los autómatas con esa regla y devuelve el valor de fitness 
correspondiente.'''
def rule_and_evolve(ca_num, individual, CAs, ca_size_1, ca_size_2):
    '''Creación de la regla de transición'''
    mi_regla = create_transition_rule(individual, P, ca_size_1, ca_size_2)
    
    '''Evolución de los autómatas con la regla creada'''
    CAs = evolved_CAs(CAs, mi_regla)

    '''Estudiar métricas para medir fitness, comparando el estado final del autómata con el estado objetivo'''
    fitness_values = []
    for ca in CAs:

        '''Media ponderada entre SSIM y Jaccard'''
        ssim = ssim_metric(ca[-1], HUN)
        jaccard = jaccard_metric(ca[-1], HUN)
        fit = 0.8 * ssim + 0.2 * jaccard

        '''counts guarda el número de elementos de cada color en el estado final del autómata, y el objetivo es penalizar el fitness (reduciéndolo a la mitad)
        si alguno de los estados tiene menos del 10% de las células totales del autómata'''
        counts = np.bincount(ca[-1].flatten(), minlength=3)
        total_cells = ca[-1].size
        if np.any(counts < (0.10 * total_cells)):
            fit = fit * 0.5  

        fitness_values.append(fit)
    
        '''metrics_accuracy.append(accuracy_metric(ca[-1], HUN))
        metrics_mse.append(mse_metric(ca[-1], HUN))
        metrics_correlation.append(correlation_metric(ca[-1], HUN))
        metrics_haussdorf.append(haussdorf_metric(ca[-1], HUN))'''

    
    '''Media de los valores de fitness obtenidos para cada autómata'''
    return np.mean(fitness_values)



'''Funcion de fitness adaptativa:
Genera CA_NUM//5 autómatas aleatoriamente, determina la regla asociada al individuo y los evoluciona con esa regla. 
Si esa regla tiene más de un 50% de fitness, se repite el proceso con CA_NUM autómátas.
Si no, se devuelve el fitness inferior a 50%. De esta forma, se evita evaluar reglas que a priori no son prometedoras
y se gana eficiencia temporal'''

def evaluate_HUN(individual):

    CAs = generate_CAs(CA_NUM//5, NUM_STATES, CA_SIZE_1, CA_SIZE_2)

    res = rule_and_evolve(CA_NUM//5, individual, CAs, CA_SIZE_1, CA_SIZE_2)

    if res < FITNESS_THRESHOLD:
        return (res,)
        
    CAs = generate_CAs(CA_NUM, NUM_STATES, CA_SIZE_1, CA_SIZE_2)
    
    res = rule_and_evolve(CA_NUM, individual, CAs, CA_SIZE_1, CA_SIZE_2)

    return (res,)

'''Funcion de mutacion personalizada: selecciona un gen al azar, lo cambia a uno distinto aleatoriamente y devuelve el individuo modificado'''
def mutChangeGen(individual):
    gen = random.randint(0, len(individual) - 1)
    
    val = individual[gen]

    choices = [i for i in range(NUM_STATES) if i != val]
    new_val = random.choice(choices)
    
    individual[gen] = new_val
    
    return individual

'''Configuración del cruce y la selección (aunque no sea necesario), selecciona la mutacion personalizada (mutChangeGen) y
registra la función de evaluacion personalizada del algoritmo '''
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", mutChangeGen)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate_HUN)

def GA():
    global NUM_MUT
    max_fitness_values = []

    '''generación aleatoria de la poblacion'''
    pop = toolbox.population(n=POP_SIZE)

    '''Evaluación de toda la población paralelamente mediante la libreria joblib y asignacion de cada fitness a su individuo'''
    print('Iniciando cálculo de fitness', flush = True)
    fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in pop)
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit
    print('Cálculo de fitness terminado', flush = True)

    
    for g in range(NGEN):
        print(f'--- Iniciando Generación {g} ---', flush=True)

        '''Estrategia clásica comentada'''
        '''Selección:
        Dejamos que el 20% de los mejores individuos pasen a la siguiente generación'''
        '''elites = list(map(toolbox.clone, tools.selBest(pop, k= round(0.2*POP_SIZE))))'''
        
        '''Hacemos selección por torneo para construir el 80% restante de la próxima generacion'''
        '''offspring = toolbox.select(pop, len(pop) - round(0.2*POP_SIZE))
        offspring = list(map(toolbox.clone, offspring))'''

        '''Cruce y mutación'''
        '''for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values'''

        '''Evaluar paralelamente individuos con fitness inválido, es decir, indiviudos cuyo fitness no se ha calculado todavía.'''
        '''invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit'''

        '''Nueva población: 
        Construimos la población nueva juntando el 20% de elitismo (elites) con el 80% de selección por torneo (offspring)'''
        '''pop[:] = elites + offspring'''

        '''Estrategia mu + lambda'''

        '''Clonamos la poblacion y mutamos cada individuo dos veces'''

        '''Ordneamos la población por fitness y nos quedamos con el 10% de los mejores individuos, que pasan a la siguiente generación'''
        pop.sort(key=lambda x: x.fitness.values[0], reverse=True)

        n_elites = int(round(0.1 * POP_SIZE))
        n_rest   = POP_SIZE - n_elites
        elites = list(map(toolbox.clone, pop[:n_elites]))

        '''Clonamos el resto de individuos que no forman parte del 10% de elitismo'''
        aux = list(map(toolbox.clone, pop[n_elites:]))
        offspring = list(map(toolbox.clone, pop[n_elites:]))

        '''Mutamos cada uno de los idnividuos'''
        for mutant in offspring:
            for i in range(NUM_MUT):
                toolbox.mutate(mutant)
            del mutant.fitness.values

        '''Calculamos los fitness de los individuos mutados paralelamente'''
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit


        '''Juntamos los individuos sin mutar con los mutados, y entre todos ellos, cogemos los mejores restantes para completar la siguiente genración junto con los 
        elites'''

        pop_aux = aux + offspring
        pop_resto = tools.selBest(pop_aux, k = n_rest)
        pop[:] = elites + pop_resto

        
        '''Descartamos el 10% de las peores reglas y las añadimos aleatoriamente -> generacion de diversidad en la población'''
        '''pop_aux1 = tools.selBest(pop_aux, k = round(POP_SIZE*0.9))
        pop_aux2 = toolbox.population(n=POP_SIZE - round(POP_SIZE*0.9))
        fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in pop_aux2)
        for ind, fit in zip(pop_aux2, fitnesses):
            ind.fitness.values = fit
            
        pop[:] = pop_aux1 + pop_aux2'''


        top = tools.selBest(pop, 3)

        '''Estrategia exploración vs explotación: ajuste dinámico del número de mutaciones. Cuanto mayor es el fitness, menor es el número de mutaciones
        realizadas, favoreciendo así la convergencia del algoritmo'''

        if (top[0].fitness.values[0] > 0.25 and top[0].fitness.values[0] < 0.45):
            NUM_MUT = 8
        elif (top[0].fitness.values[0] > 0.45 and top[0].fitness.values[0] < 0.6):
            NUM_MUT = 6
        elif (top[0].fitness.values[0] > 0.6 and top[0].fitness.values[0] < 0.75):
            NUM_MUT = 4
        elif (top[0].fitness.values[0] > 0.75):
            NUM_MUT = 2


        print(f"--- Gen {g} Completada: Max Fitness (SSIM)= {top[0].fitness.values[0]:.2f}", flush=True)
     

        max_fitness_values.append(top[0].fitness.values[0])
        print(top[0])
        if top[0].fitness.values[0] >= STOP_CONDITION: #Estudiar condición de parada
            print(f"Parado en la generación {g} con fitness {top[0].fitness.values[0]}")
            return top

    return tools.selBest(pop, 3)

In [ ]:
'''Código de colores'''
AZUL_CLARO   = "\033[94m" # Para el estado azul
BLANCO_BRILLANTE = "\033[97m" # Para el estado blanco
ROJO_BRILLANTE   = "\033[91m" # Para el estado rojo
VERDE_BRILLANTE = "\033[92m" # Para el estado verde
RESET      = "\033[0m"  # Para resetear el color al final

'''Funcion que, dada una regla, la muestra gráfciamente con las configuraciones de vecinos y los colores correspondientes'''
def dibujar_regla(regla):
    simbolo = "■" 

    # Mapeo: 0=Rojo, 1=Blanco, 2=Verde (Bandera Húngara)
    # Ajusta los colores si usas otra bandera
    simbolos_map = { 
        '0': f"{ROJO_BRILLANTE}{simbolo}{RESET}", 
        '1': f"{BLANCO_BRILLANTE}{simbolo}{RESET}",
        '2': f"{VERDE_BRILLANTE}{simbolo}{RESET}" 
    }
    
    print(f"--- Catálogo de Reglas Von Neumann (5 Vecinos -> Resultado) ---")
    print("Formato visual:")
    print("  N  ")
    print("W C E  ->  Resultado")
    print("  S  ")
    print("-" * 40)
    
    # Von Neumann son 5 celdas
    longitud_vecindad = 5 

    for indice in range(len(regla)):
        
        # 1. Convertir índice a base 3 (o 2 si usas austriaca), rellenando a 5 dígitos
        config_str = np.base_repr(indice, base=NUM_STATES).zfill(longitud_vecindad)
        
        # 2. Mapear cada posición según el orden [Centro, Norte, Este, Sur, Oeste]
        
        C = simbolos_map[config_str[0]] # Centro
        N = simbolos_map[config_str[1]] # Norte
        E = simbolos_map[config_str[2]] # Este
        S = simbolos_map[config_str[3]] # Sur
        W = simbolos_map[config_str[4]] # Oeste
        
        # 3. Resultado
        resultado = regla[indice]
        res_visual = simbolos_map[str(resultado)]
        
        # 4. Imprimir en formato bloque (Cross layout)
        # Usamos caracteres invisibles para alinear o espacios simples
        print(f"Índice {indice:03d}:")
        print(f"    {N}    ")       # Línea superior (Norte)
        print(f"  {W} {C} {E}  ->  {res_visual}") # Línea media (Oeste, Centro, Este) -> Resultado
        print(f"    {S}    ")       # Línea inferior (Sur)
        print("-" * 20)             # Separador


'''Llama al algoritmo y hace un test con las 3 mejores reglas'''
if __name__ == "__main__":
    top3 = GA()
    CA_NUM_TEST = 250
    for individual in top3:
        regla = np.array(individual)
        dibujar_regla(regla)
        
        CAs = generate_CAs(CA_NUM_TEST, NUM_STATES, CA_SIZE_1, CA_SIZE_2)

        '''rule_dict = gen_rule_dict(individual)'''
        mi_regla = create_transition_rule(individual, P, CA_SIZE_1, CA_SIZE_2)

        '''Evolucion de los autómatas de prueba con la regla (ahora sí, en paralelo)'''
        CAs = Parallel(n_jobs=-1)(delayed(cpl.evolve2d)(CAs[i], timesteps=CA_TIMESTEPS, neighbourhood = "von Neumann", apply_rule=mi_regla) for i in range(len(CAs)))

        '''Estudio del rendimiento de la regla'''
        metrics = []
        for ca in CAs:
            metrics.append(ssim_metric(ca[-1], HUN))
                
        porcentaje_exito = (np.sum(metrics) / CA_NUM_TEST) * 100
        print("\n" + "*"*30)
        print(f"  RENDIMIENTO DE LA REGLA:")
        print(f"  FITNESS:  {porcentaje_exito:.2f}%")
        print("*"*30 + "\n")

        
        '''Graficación de la evolucion de los autómatas'''
        colores_ternarios = ['blue', 'white', 'red'] 
        cmap_personal = ListedColormap(colores_ternarios)
        
        for i in range(len(CAs)):
            plt.figure(figsize=(8, 4))
            plt.imshow(CAs[i][-1], cmap=cmap_personal, interpolation='nearest', aspect='auto')
            plt.xlabel("Celda")
            plt.ylabel("Tiempo")
            plt.title(f"Evolución del autómata CA {i}")
            plt.show()

In [ ]:

regla_1 = [0, 0, 0, 0, 1, 2, 0, 1, 1, 0, 1, 1, 0, 0, 2, 0, 1, 2, 2, 1, 1, 1, 1, 1, 2, 2, 1, 
           0, 0, 2, 1, 1, 1, 2, 1, 0, 2, 1, 2, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 2, 2, 2, 
           1, 1, 0, 0, 0, 0, 2, 1, 2, 2, 1, 1, 1, 2, 2, 1, 2, 1, 2, 2, 0, 1, 0, 1, 1, 0, 2, 
           0, 1, 0, 1, 0, 1, 0, 0, 2, 1, 0, 0, 0, 0, 2, 0, 1, 1, 1, 1, 0, 1, 2, 0, 1, 2, 2, 
           0, 0, 2, 2, 0, 0, 2, 1, 2, 1, 0, 2, 0, 0, 2, 2, 0, 2, 0, 0, 0, 1, 0, 0, 1, 0, 2, 
           0, 2, 1, 0, 1, 0, 0, 1, 1, 1, 1, 2, 0, 2, 2, 1, 2, 2, 1, 2, 0, 0, 2, 0, 0, 2, 1, 
           2, 2, 1, 0, 1, 0, 1, 2, 2, 1, 1, 2, 2, 0, 1, 0, 1, 2, 1, 1, 0, 2, 2, 0, 1, 0, 2, 
           2, 0, 1, 0, 2, 1, 2, 2, 0, 1, 1, 0, 2, 1, 1, 0, 1, 1, 2, 1, 0, 2, 1, 0, 2, 2, 2, 
           0, 2, 0, 1, 0, 2, 2, 0, 2, 1, 2, 2, 2, 1, 1, 1, 0, 2, 1, 1, 1, 0, 1, 1, 1, 2, 2] #0.43 SSIM

#A partir de generación 140, la mejora era muy poco significativa, se decidió detener el algoritmo y probar con distintas alternativas.
#Implementación: estrategia mu + lambda, 5 mutaciones siempre, SSIM 100%, parada si se alcanza fitness 0.75, fitness adaptativo, selección global entre 
#mutados y no mutados. Parámetros utilizados:

regla_2 = [0, 0, 1, 0, 0, 0, 0, 0, 2, 2, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 2, 0, 0, 0, 2, 1, 2, 
           1, 0, 1, 0, 2, 0, 2, 0, 2, 0, 0, 2, 0, 2, 2, 0, 1, 1, 0, 1, 2, 0, 1, 2, 2, 2, 2, 
           1, 0, 0, 0, 2, 2, 1, 1, 1, 1, 1, 0, 2, 1, 0, 0, 2, 1, 1, 2, 1, 0, 2, 0, 1, 1, 1, 
           0, 0, 0, 0, 1, 0, 2, 2, 1, 0, 0, 1, 2, 0, 0, 1, 0, 2, 2, 0, 0, 0, 0, 1, 2, 2, 1, 
           1, 1, 2, 0, 2, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 2, 2, 0, 1, 0, 1, 0, 1, 2, 0, 2,
           1, 0, 2, 0, 0, 1, 1, 2, 1, 1, 2, 1, 0, 1, 0, 2, 1, 1, 1, 2, 0, 0, 2, 0, 1, 0, 1,
           0, 1, 1, 0, 0, 1, 2, 1, 2, 0, 0, 1, 1, 0, 1, 1, 0, 2, 0, 2, 1, 2, 0, 0, 2, 2, 2, 
           1, 0, 0, 1, 1, 2, 0, 1, 2, 1, 1, 1, 1, 0, 1, 1, 2, 2, 2, 2, 1, 1, 1, 0, 2, 2, 2, 
           2, 2, 2, 1, 2, 1, 2, 1, 2, 0, 0, 0, 0, 2, 2, 0, 1, 2, 2, 2, 1, 2, 0, 0, 1, 2, 2] #0.62 SSIM 60% + JACCARD 40% ---> Mitad verde mitad rojo, hay que penalizar que no haya blanco). Reducimos ventana SSIM a 5, 
                                                                                                #implementamos mutaciones dinámicas y dejamos pasar al 20% de las mejores reglas (elitismo).



regla_3 = [0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 1, 1, 2, 2, 2, 0, 0, 2, 1, 0, 0, 0, 0, 2, 
           1, 2, 0, 1, 1, 1, 0, 1, 0, 1, 2, 0, 1, 1, 0, 1, 1, 2, 1, 1, 1, 0, 2, 1, 0, 2, 2, 
           1, 0, 0, 2, 1, 0, 1, 1, 0, 2, 1, 2, 1, 1, 2, 1, 1, 2, 0, 2, 0, 0, 2, 2, 0, 2, 0, 
           2, 1, 2, 0, 0, 2, 2, 1, 1, 0, 2, 1, 0, 1, 2, 1, 0, 2, 2, 2, 1, 0, 1, 1, 0, 1, 2, 
           1, 1, 1, 1, 1, 0, 0, 2, 0, 1, 1, 1, 1, 1, 1, 2, 1, 0, 0, 2, 1, 0, 1, 0, 2, 1, 2, 
           1, 2, 0, 0, 2, 1, 1, 2, 1, 1, 0, 2, 1, 1, 1, 0, 2, 0, 2, 2, 2, 1, 1, 2, 1, 1, 2, 
           1, 2, 1, 1, 0, 0, 1, 0, 2, 1, 1, 2, 1, 0, 2, 2, 0, 1, 0, 2, 2, 1, 1, 1, 2, 1, 2, 
           0, 1, 1, 2, 0, 1, 1, 1, 2, 0, 0, 1, 1, 1, 1, 0, 2, 2, 1, 1, 2, 1, 0, 1, 1, 2, 2, 
           0, 1, 0, 1, 2, 0, 1, 2, 2, 0, 2, 1, 1, 1, 2, 1, 1, 0, 1, 2, 0, 2, 1, 1, 2, 0, 2] #0.76 SSIM 80% + JACCARD 20% ---> Para evitar lo ocurrido en la regla 2, se pnealiza que haya menos de un 10% de cualquier
                                            #color en el estado final del autómata. Mantenemos ajuste dinámico de mutaciones y ventana SSIM de 5. Dejamos pasar al 10% de las mejores reglas (elitismo).

regla_4 = [0, 0, 0, 0, 0, 0, 1, 2, 1, 0, 0, 1, 0, 1, 1, 0, 2, 2, 0, 2, 1, 1, 0, 0, 1, 1, 2, 
           0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 2, 1, 1, 1, 0, 2, 1, 2, 0, 2, 
           0, 0, 1, 2, 2, 1, 1, 1, 0, 2, 1, 1, 1, 1, 2, 2, 1, 2, 0, 2, 1, 0, 2, 2, 0, 2, 0, 
           1, 1, 2, 0, 0, 2, 2, 1, 1, 2, 0, 1, 0, 1, 0, 2, 2, 2, 0, 2, 2, 0, 1, 2, 0, 2, 2, 
           1, 1, 1, 1, 1, 1, 0, 2, 0, 1, 1, 1, 1, 1, 0, 2, 1, 0, 1, 2, 0, 1, 1, 2, 2, 1, 2, 
           1, 1, 0, 0, 1, 2, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 2, 0, 2, 2, 2, 1, 1, 2, 1, 1, 0, 
           1, 2, 1, 1, 1, 0, 1, 0, 2, 1, 1, 2, 1, 0, 1, 0, 1, 1, 1, 2, 1, 0, 1, 1, 2, 1, 2, 
           0, 0, 1, 1, 0, 2, 1, 1, 2, 0, 0, 2, 2, 1, 0, 2, 2, 2, 1, 1, 2, 1, 0, 1, 1, 2, 2, 
           1, 1, 0, 0, 2, 0, 1, 0, 2, 0, 2, 2, 0, 2, 0, 1, 1, 0, 0, 2, 1, 2, 2, 1, 2, 0, 2] #0.78 opcion 1 SSIM 80% + JACCARD 20%

regla_5 = [0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 1, 1, 2, 2, 2, 0, 0, 2, 1, 0, 0, 0, 0, 2, 
           1, 2, 0, 0, 1, 1, 0, 1, 0, 1, 2, 0, 1, 1, 0, 1, 1, 2, 1, 1, 1, 2, 2, 1, 0, 2, 2, 
           1, 0, 0, 2, 1, 1, 1, 1, 0, 2, 1, 2, 1, 1, 2, 1, 1, 2, 0, 2, 0, 0, 0, 2, 0, 2, 0, 
           1, 1, 2, 0, 0, 2, 2, 1, 2, 0, 2, 1, 0, 1, 2, 1, 0, 2, 0, 2, 2, 0, 1, 1, 0, 1, 0, 
           1, 1, 1, 1, 1, 0, 2, 2, 0, 1, 1, 1, 1, 1, 1, 2, 1, 0, 0, 2, 1, 0, 1, 0, 2, 1, 2, 
           0, 2, 0, 0, 2, 1, 1, 2, 1, 1, 0, 1, 1, 1, 1, 0, 2, 0, 2, 2, 2, 1, 1, 2, 1, 1, 2, 
           1, 2, 1, 1, 1, 1, 1, 0, 2, 1, 1, 2, 1, 0, 2, 2, 0, 1, 0, 0, 0, 1, 1, 1, 2, 1, 2, 
           0, 1, 1, 2, 0, 1, 1, 1, 2, 0, 0, 2, 1, 1, 1, 0, 2, 2, 1, 1, 2, 2, 0, 1, 1, 2, 2, 
           0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 2, 1, 1, 1, 2, 1, 1, 0, 1, 2, 0, 2, 1, 1, 2, 0, 2]
#0.78 opcion 2 SSIM 80% + JACCARD 20%


num_automata = 50
CA_SIZE_LIST_1 = [18]
CA_SIZE_LIST_2 = [30]
CA_TIMESTEPS_LIST = [150]
reglas = [regla_4, regla_5]

for individual in reglas:
    dibujar_regla(individual)
    for j in range(len(CA_SIZE_LIST_1)):
        HUN = np.zeros((CA_SIZE_LIST_1[j], CA_SIZE_LIST_2[j]), dtype=int)
        h = CA_SIZE_LIST_1[j] // 3
        HUN[0:h, :] = 0     
        HUN[h:2*h, :] = 1   
        HUN[2*h:, :] = 2
        
        CAs = generate_CAs(num_automata, NUM_STATES, CA_SIZE_LIST_1[j], CA_SIZE_LIST_2[j])
    
        rule_dict = gen_rule_dict(individual)
        mi_regla = create_transition_rule(individual, P, CA_SIZE_LIST_1[j], CA_SIZE_LIST_2[j])

        CAs = Parallel(n_jobs=-1)(delayed(cpl.evolve2d)(CAs[i], timesteps=CA_TIMESTEPS_LIST[j], neighbourhood = "von Neumann", apply_rule=mi_regla) for i in range(len(CAs)))
    
        #Estudiamos los resultados
        common_cells = []
        for ca in CAs:
            common_cells.append(np.sum(ca[-1] == HUN)/(CA_SIZE_LIST_1[j]*CA_SIZE_LIST_2[j]))
                
        porcentaje_exito = (np.sum(common_cells) / num_automata) * 100
        print("\n" + "*"*30)
        print(f"  RENDIMIENTO DE LA REGLA:")
        print(f"  ACCURACY:  {porcentaje_exito:.2f}%")
        print("*"*30 + "\n")
    
        # 4. Definir colores y graficar
        colores_ternarios = ['red', 'white', 'green'] 
        cmap_personal = ListedColormap(colores_ternarios)
        
        for i in range(len(CAs)):
            plt.figure(figsize=(8, 4))
            plt.imshow(CAs[i][-1], cmap=cmap_personal, interpolation='nearest', aspect='auto')
            plt.xlabel("Celda")
            plt.ylabel("Tiempo")
            plt.title(f"Evolución del autómata CA {i}")
            plt.show()
    

In [ ]:
#Visualización de la evolución de un autómata aleatorio con la mejor regla encontrada

rc('animation', html='jshtml')
CA_SIZE_1 = 18
CA_SIZE_2 = 30
CA_TIMESTEPS = 200

colores = ['#CE2939', '#FFFFFF', '#477050']
mi_cmap = ListedColormap(colores)
regla_1 = [0, 0, 0, 0, 0, 0, 1, 2, 1, 0, 0, 1, 0, 1, 1, 0, 2, 2, 0, 2, 1, 1, 0, 0, 1, 1, 2, 
           0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 2, 1, 1, 1, 0, 2, 1, 2, 0, 2, 
           0, 0, 1, 2, 2, 1, 1, 1, 0, 2, 1, 1, 1, 1, 2, 2, 1, 2, 0, 2, 1, 0, 2, 2, 0, 2, 0, 
           1, 1, 2, 0, 0, 2, 2, 1, 1, 2, 0, 1, 0, 1, 0, 2, 2, 2, 0, 2, 2, 0, 1, 2, 0, 2, 2, 
           1, 1, 1, 1, 1, 1, 0, 2, 0, 1, 1, 1, 1, 1, 0, 2, 1, 0, 1, 2, 0, 1, 1, 2, 2, 1, 2, 
           1, 1, 0, 0, 1, 2, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 2, 0, 2, 2, 2, 1, 1, 2, 1, 1, 0, 
           1, 2, 1, 1, 1, 0, 1, 0, 2, 1, 1, 2, 1, 0, 1, 0, 1, 1, 1, 2, 1, 0, 1, 1, 2, 1, 2, 
           0, 0, 1, 1, 0, 2, 1, 1, 2, 0, 0, 2, 2, 1, 0, 2, 2, 2, 1, 1, 2, 1, 0, 1, 1, 2, 2, 
           1, 1, 0, 0, 2, 0, 1, 0, 2, 0, 2, 2, 0, 2, 0, 1, 1, 0, 0, 2, 1, 2, 2, 1, 2, 0, 2] #0.78 opcion 1 SSIM 80% + JACCARD 20%

regla_2 = [0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 1, 1, 2, 2, 2, 0, 0, 2, 1, 0, 0, 0, 0, 2, 
           1, 2, 0, 0, 1, 1, 0, 1, 0, 1, 2, 0, 1, 1, 0, 1, 1, 2, 1, 1, 1, 2, 2, 1, 0, 2, 2, 
           1, 0, 0, 2, 1, 1, 1, 1, 0, 2, 1, 2, 1, 1, 2, 1, 1, 2, 0, 2, 0, 0, 0, 2, 0, 2, 0, 
           1, 1, 2, 0, 0, 2, 2, 1, 2, 0, 2, 1, 0, 1, 2, 1, 0, 2, 0, 2, 2, 0, 1, 1, 0, 1, 0, 
           1, 1, 1, 1, 1, 0, 2, 2, 0, 1, 1, 1, 1, 1, 1, 2, 1, 0, 0, 2, 1, 0, 1, 0, 2, 1, 2, 
           0, 2, 0, 0, 2, 1, 1, 2, 1, 1, 0, 1, 1, 1, 1, 0, 2, 0, 2, 2, 2, 1, 1, 2, 1, 1, 2, 
           1, 2, 1, 1, 1, 1, 1, 0, 2, 1, 1, 2, 1, 0, 2, 2, 0, 1, 0, 0, 0, 1, 1, 1, 2, 1, 2, 
           0, 1, 1, 2, 0, 1, 1, 1, 2, 0, 0, 2, 1, 1, 1, 0, 2, 2, 1, 1, 2, 2, 0, 1, 1, 2, 2, 
           0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 2, 1, 1, 1, 2, 1, 1, 0, 1, 2, 0, 2, 1, 1, 2, 0, 2] #0.78 opcion 2 SSIM 80% + JACCARD 20%


dibujar_regla(regla_1)

# 2. Asumiendo que 'best_ind' es tu mejor individuo salido del GA
# Creamos la función de la regla
regla_ganadora = create_transition_rule(regla_1, 1.0, CA_SIZE_1, CA_SIZE_2)


# 3. Inicializamos UN solo autómata para probar (con bordes fijos)
inicial = np.random.randint(0, NUM_STATES, size=(CA_SIZE_1, CA_SIZE_2))
inicial[0, :] = 0
inicial[-1, :] = 2
inicial = np.expand_dims(inicial, axis=0)

# 4. Evolucionamos (esto genera el historial completo que necesita la animación)
# cpl.evolve2d devuelve un array de forma (timesteps, alto, ancho)
ca = cpl.evolve2d(
    inicial, 
    timesteps=CA_TIMESTEPS, 
    neighbourhood="von Neumann", 
    apply_rule=regla_ganadora
)

# 5. VISUALIZACIÓN DIRECTA con cpl
cpl.plot2d_animate(ca, colormap=mi_cmap)




In [ ]:
'''Extraccion de datos de la salida por consola del algoritmo para construir gráfica de evolución del fitness. Código hecho con Gemini'''

log_data = """Incluir salida"""

# 1. Extraer datos usando Expresiones Regulares (Regex)
# Buscamos el patrón "Gen [NUMERO] ... Fitness = [DECIMAL]"
patron = r"Gen (\d+) Completada: Max Fitness \(SSIM\)= (\d+\.\d+)"
coincidencias = re.findall(patron, log_data)

# 2. Convertir los textos extraídos a listas numéricas
generaciones = [int(dato[0]) for dato in coincidencias]
fitness_values = [float(dato[1]) for dato in coincidencias]

# 3. Configuración de la Gráfica
plt.figure(figsize=(12, 6)) # Tamaño de la imagen
plt.plot(generaciones, fitness_values, 
         linewidth=2, 
         color='#1f77b4', # Azul estándar
         label='Mejor Individuo')

# Añadir puntos en los momentos donde cambia el fitness (escalones)
# Esto ayuda a visualizar cuándo ocurrieron las mutaciones exitosas
plt.scatter(generaciones, fitness_values, color='red', s=10, zorder=5)

# 4. Etiquetas y Estilo
plt.title('Evolución del Fitness (Algoritmo Genético)', fontsize=14)
plt.xlabel('Generación', fontsize=12)
plt.ylabel('Max Fitness', fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.legend()

# Mostrar valor final
max_val = max(fitness_values)
max_gen = generaciones[fitness_values.index(max_val)]
plt.annotate(f'Max: {max_val}', xy=(max_gen, max_val), xytext=(max_gen-50, max_val+0.01),
             arrowprops=dict(facecolor='black', shrink=0.05))

# 5. Mostrar
plt.tight_layout()
plt.show()